In [14]:
import requests
import pandas as pd

url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
resposta = requests.get(url_ipca)
ipca = pd.DataFrame(resposta.json())

ipca["data"] = pd.to_datetime(ipca["data"], dayfirst=True)
ipca = ipca[ipca["data"] >= "2023-01-01"]

print(ipca)

          data  valor
516 2023-01-01   0.53
517 2023-02-01   0.84
518 2023-03-01   0.71
519 2023-04-01   0.61
520 2023-05-01   0.23
521 2023-06-01  -0.08
522 2023-07-01   0.12
523 2023-08-01   0.23
524 2023-09-01   0.26
525 2023-10-01   0.24
526 2023-11-01   0.28
527 2023-12-01   0.56
528 2024-01-01   0.42
529 2024-02-01   0.83
530 2024-03-01   0.16
531 2024-04-01   0.38
532 2024-05-01   0.46
533 2024-06-01   0.21
534 2024-07-01   0.38
535 2024-08-01  -0.02
536 2024-09-01   0.44
537 2024-10-01   0.56
538 2024-11-01   0.39
539 2024-12-01   0.52
540 2025-01-01   0.16
541 2025-02-01   1.31
542 2025-03-01   0.56
543 2025-04-01   0.43
544 2025-05-01   0.26
545 2025-06-01   0.24
546 2025-07-01   0.26
547 2025-08-01  -0.11
548 2025-09-01   0.48
549 2025-10-01   0.09
550 2025-11-01   0.18
551 2025-12-01   0.33
552 2026-01-01   0.33
553 2026-02-01   0.70
554 2026-03-01   0.88
555 2026-04-01   0.67
556 2026-05-01   0.58
557 2026-06-01   0.16


In [15]:
import yfinance as yf

ewz = yf.download("EWZ", start="2023-01-01", end="2026-08-06", progress=True)
ewz = ewz[["Close"]].reset_index()
ewz.columns = ["data", "close"]
ewz["data"] = ewz["data"].dt.strftime("%Y-%m-%d")
ewz["retorno_pct"] = ewz["close"].pct_change() * 100

ewz.to_csv("../data/ewz_precos.csv", index=False)
print(ewz.tail())

[*********************100%***********************]  1 of 1 completed

           data      close  retorno_pct
895  2026-07-30  36.529999     2.988434
896  2026-07-31  36.650002     0.328505
897  2026-08-03  36.419998    -0.627567
898  2026-08-04  36.090000    -0.906090
899  2026-08-05  36.110001     0.055418


In [16]:
import pandas as pd

cpi_completo = pd.read_csv("../data/eventos_completo.csv")
ipca_completo = pd.read_csv("../data/eventos_ipca_completo.csv")

eventos_todos = pd.concat([cpi_completo, ipca_completo], ignore_index=True)
eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)

print(eventos_todos["indicador"].value_counts())

indicador
CPI_EUA    39
IPCA_BR    34
Name: count, dtype: int64


In [17]:
import requests
import pandas as pd

url_selic = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
    "?formato=json&dataInicial=01/01/2023&dataFinal=06/08/2026"
)

resposta = requests.get(url_selic)
print(f"Status code: {resposta.status_code}")

selic = pd.DataFrame(resposta.json())
selic["data"] = pd.to_datetime(selic["data"], dayfirst=True)

print(selic.tail(15))
print(f"\nTotal de linhas: {len(selic)}")

Status code: 200
           data  valor
1299 2026-07-23  14.25
1300 2026-07-24  14.25
1301 2026-07-25  14.25
1302 2026-07-26  14.25
1303 2026-07-27  14.25
1304 2026-07-28  14.25
1305 2026-07-29  14.25
1306 2026-07-30  14.25
1307 2026-07-31  14.25
1308 2026-08-01  14.25
1309 2026-08-02  14.25
1310 2026-08-03  14.25
1311 2026-08-04  14.25
1312 2026-08-05  14.25
1313 2026-08-06  14.00

Total de linhas: 1314


In [18]:
selic = selic.sort_values("data").reset_index(drop=True)
selic["valor_anterior"] = selic["valor"].shift(1)

decisoes = selic[selic["valor"] != selic["valor_anterior"]].copy()
decisoes = decisoes.dropna(subset=["valor_anterior"])

print(decisoes[["data", "valor_anterior", "valor"]])
print(f"\nTotal de decisões: {len(decisoes)}")

           data valor_anterior  valor
214  2023-08-03          13.75  13.25
263  2023-09-21          13.25  12.75
305  2023-11-02          12.75  12.25
347  2023-12-14          12.25  11.75
396  2024-02-01          11.75  11.25
445  2024-03-21          11.25  10.75
494  2024-05-09          10.75  10.50
627  2024-09-19          10.50  10.75
676  2024-11-07          10.75  11.25
711  2024-12-12          11.25  12.25
760  2025-01-30          12.25  13.25
809  2025-03-20          13.25  14.25
858  2025-05-08          14.25  14.75
900  2025-06-19          14.75  15.00
1173 2026-03-19          15.00  14.75
1215 2026-04-30          14.75  14.50
1264 2026-06-18          14.50  14.25
1313 2026-08-06          14.25  14.00

Total de decisões: 18


In [19]:
url_selic_exp = (
    "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/"
    "ExpectativasMercadoSelic?$top=20&$format=json"
    "&$filter=Data%20ge%20%272023-01-01%27"
    "&$orderby=Data%20desc"
)

resposta = requests.get(url_selic_exp)
print(f"Status code: {resposta.status_code}")

expectativas_selic = pd.DataFrame(resposta.json()["value"])
print(expectativas_selic.columns.tolist())
print(expectativas_selic.head(10))

Status code: 200
['Indicador', 'Data', 'Reuniao', 'Media', 'Mediana', 'DesvioPadrao', 'Minimo', 'Maximo', 'numeroRespondentes', 'baseCalculo']
  Indicador        Data  Reuniao    Media  Mediana  DesvioPadrao  Minimo  \
0     Selic  2026-08-07  R5/2028  10.8333   10.750        1.1242    9.00   
1     Selic  2026-08-07  R5/2028  10.8333   10.750        1.1242    9.00   
2     Selic  2026-08-07  R4/2028  11.3513   11.375        0.9451    9.00   
3     Selic  2026-08-07  R4/2028  11.0776   11.000        0.9004    9.00   
4     Selic  2026-08-07  R3/2028  11.4778   11.500        0.8833    9.25   
5     Selic  2026-08-07  R3/2028  11.2422   11.250        0.7843    9.50   
6     Selic  2026-08-07  R2/2028  11.6687   11.750        0.8356    9.50   
7     Selic  2026-08-07  R2/2028  11.4545   11.500        0.7030   10.00   
8     Selic  2026-08-07  R1/2028  11.8799   12.000        0.8059    9.75   
9     Selic  2026-08-07  R1/2028  11.6618   11.875        0.6639   10.00   

   Maximo  numeroRes

In [20]:
anos = [2022, 2023, 2024, 2025, 2026]
partes = []

for ano in anos:
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/"
        "ExpectativasMercadoSelic?$top=10000&$format=json"
        f"&$filter=Data%20ge%20%27{ano}-01-01%27%20and%20Data%20le%20%27{ano}-12-31%27%20and%20baseCalculo%20eq%200"
        "&$orderby=Data%20asc"
    )
    resposta = requests.get(url)
    parte = pd.DataFrame(resposta.json()["value"])
    print(f"{ano}: {len(parte)} linhas")
    partes.append(parte)

expectativas_selic = pd.concat(partes, ignore_index=True)
expectativas_selic["Data"] = pd.to_datetime(expectativas_selic["Data"])

print(f"\nTotal combinado: {len(expectativas_selic)}")
print(f"Data mínima: {expectativas_selic['Data'].min()}")
print(f"Data máxima: {expectativas_selic['Data'].max()}")

2022: 4016 linhas
2023: 3984 linhas
2024: 4048 linhas
2025: 4032 linhas
2026: 2400 linhas

Total combinado: 18480
Data mínima: 2022-01-03 00:00:00
Data máxima: 2026-08-07 00:00:00


In [21]:
expectativas_selic["ano_reuniao"] = expectativas_selic["Reuniao"].str.extract(r"/(\d+)").astype(int)
expectativas_selic["numero_reuniao"] = expectativas_selic["Reuniao"].str.extract(r"R(\d+)").astype(int)

print(expectativas_selic[["Data", "Reuniao", "ano_reuniao", "numero_reuniao", "Mediana"]].head())

        Data  Reuniao  ano_reuniao  numero_reuniao  Mediana
0 2022-01-03  R1/2022         2022               1    10.75
1 2022-01-03  R2/2022         2022               2    11.75
2 2022-01-03  R3/2022         2022               3    11.75
3 2022-01-03  R4/2022         2022               4    11.75
4 2022-01-03  R5/2022         2022               5    11.75


In [22]:
resultados = []

for _, decisao in decisoes.iterrows():
    data_decisao = decisao["data"]

    candidatos = expectativas_selic[expectativas_selic["Data"] < data_decisao]
    if candidatos.empty:
        continue

    ultima_data_coleta = candidatos["Data"].max()
    candidatos_ultima_data = candidatos[candidatos["Data"] == ultima_data_coleta]

    candidatos_ultima_data = candidatos_ultima_data.sort_values(["ano_reuniao", "numero_reuniao"])
    proxima_reuniao = candidatos_ultima_data.iloc[0]

    resultados.append({
        "data": data_decisao.strftime("%Y-%m-%d"),
        "actual": decisao["valor"],
        "forecast": proxima_reuniao["Mediana"]
    })

eventos_selic = pd.DataFrame(resultados)
eventos_selic["indicador"] = "Selic_BR"
eventos_selic = eventos_selic[["indicador", "data", "actual", "forecast"]]

print(eventos_selic)

   indicador        data actual  forecast
0   Selic_BR  2023-08-03  13.25     13.50
1   Selic_BR  2023-09-21  12.75     12.75
2   Selic_BR  2023-11-02  12.25     12.25
3   Selic_BR  2023-12-14  11.75     11.75
4   Selic_BR  2024-02-01  11.25     11.25
5   Selic_BR  2024-03-21  10.75     10.75
6   Selic_BR  2024-05-09  10.50     10.50
7   Selic_BR  2024-09-19  10.75     10.75
8   Selic_BR  2024-11-07  11.25     11.25
9   Selic_BR  2024-12-12  12.25     12.00
10  Selic_BR  2025-01-30  13.25     13.25
11  Selic_BR  2025-03-20  14.25     14.25
12  Selic_BR  2025-05-08  14.75     14.75
13  Selic_BR  2025-06-19  15.00     14.75
14  Selic_BR  2026-03-19  14.75     14.75
15  Selic_BR  2026-04-30  14.50     14.50
16  Selic_BR  2026-06-18  14.25     14.25
17  Selic_BR  2026-08-06  14.00     14.00


In [23]:
eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_atualizado = pd.concat([eventos_principal, eventos_selic], ignore_index=True)
eventos_atualizado = eventos_atualizado.drop_duplicates()
eventos_atualizado.to_csv("../data/eventos.csv", index=False)

print(eventos_atualizado["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
Selic_BR       36
IPCA_BR        34
Name: count, dtype: int64


In [24]:
import yfinance as yf

usdbrl = yf.download("BRL=X", start="2023-01-01", end="2026-08-06", progress=True)
usdbrl = usdbrl[["Close"]].reset_index()
usdbrl.columns = ["data", "close"]
usdbrl["data"] = usdbrl["data"].dt.strftime("%Y-%m-%d")
usdbrl["retorno_pct"] = usdbrl["close"].pct_change() * 100

usdbrl.to_csv("../data/usdbrl_precos.csv", index=False)
print(usdbrl.tail())

[*********************100%***********************]  1 of 1 completed

           data   close  retorno_pct
928  2026-07-30  5.1271    -0.194667
929  2026-07-31  5.0781    -0.955702
930  2026-08-03  5.0727    -0.106343
931  2026-08-04  5.1029     0.595344
932  2026-08-05  5.1437     0.799547


In [25]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

eventos_selic_completo = eventos_atualizado[eventos_atualizado["indicador"] == "Selic_BR"].copy()
eventos_selic_completo["actual"] = pd.to_numeric(eventos_selic_completo["actual"], errors="coerce")
eventos_selic_completo["forecast"] = pd.to_numeric(eventos_selic_completo["forecast"], errors="coerce")

print(eventos_selic_completo[["actual", "forecast"]].dtypes)

eventos_selic_completo = calcular_surpresa(eventos_selic_completo)
eventos_selic_completo = calcular_ian(eventos_selic_completo, termos_busca=["Selic", "juros"], geo="BR")
eventos_selic_completo = calcular_ice(
    eventos_selic_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(eventos_selic_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

actual      float64
forecast    float64
dtype: object
         data  surpresa_zscore       IAN       ICE
0  2023-08-03        -2.437908  0.815385  0.213059
1  2023-08-03        -2.437908  0.815385  0.213059
2  2023-09-21         0.000000  0.415385  0.171851
3  2023-09-21         0.000000  0.415385  0.171851
4  2023-11-02         0.000000  0.000000  0.078348
5  2023-11-02         0.000000  0.000000  0.078348
6  2023-12-14         0.000000  0.200000  0.044997
7  2023-12-14         0.000000  0.200000  0.044997
8  2024-02-01         0.000000  0.169231  0.119899
9  2024-02-01         0.000000  0.169231  0.119899
10 2024-03-21         0.000000  0.261538 -0.059981
11 2024-03-21         0.000000  0.261538 -0.059981
12 2024-05-09         0.000000  0.153846  0.101276
13 2024-05-09         0.000000  0.153846  0.101276
14 2024-09-19         0.000000  0.446154 -0.335207
15 2024-09-19         0.000000  0.446154 -0.335207
16 2024-11-07         0.000000  0.430769  0.334686
17 2024-11-07         0.0000

In [26]:
eventos_selic_completo.to_csv("../data/eventos_selic_completo.csv", index=False)

In [27]:
import pandas as pd

datas_ipca = pd.read_csv("../data/ipca.csv")
print(datas_ipca.columns.tolist())
print(datas_ipca.head())

['Release date;Time;Actual;Forecast;Previous']
       Release date;Time;Actual;Forecast;Previous
Aug 11              2026 (Jul);09:00;;0.03%;0.16%
Jul 10         2026 (Jun);09:00;0.16%;0.31%;0.58%
Jun 12         2026 (May);09:00;0.58%;0.53%;0.67%
May 12         2026 (Apr);09:00;0.67%;0.70%;0.88%
Apr 10         2026 (Mar);09:00;0.88%;0.77%;0.70%


In [28]:
datas_ipca = pd.read_csv("../data/ipca.csv", sep=";")
print(datas_ipca.columns.tolist())
print(datas_ipca.head())

['Release date', 'Time', 'Actual', 'Forecast', 'Previous']
         Release date   Time Actual Forecast Previous
0  Aug 11, 2026 (Jul)  09:00    NaN    0.03%    0.16%
1  Jul 10, 2026 (Jun)  09:00  0.16%    0.31%    0.58%
2  Jun 12, 2026 (May)  09:00  0.58%    0.53%    0.67%
3  May 12, 2026 (Apr)  09:00  0.67%    0.70%    0.88%
4  Apr 10, 2026 (Mar)  09:00  0.88%    0.77%    0.70%


In [29]:
datas_ipca = datas_ipca.dropna(subset=["Actual"])

datas_ipca["data_divulgacao"] = pd.to_datetime(
    datas_ipca["Release date"].str.split("(").str[0].str.strip(),
    format="%b %d, %Y"
)

mes_ref_texto = datas_ipca["Release date"].str.extract(r"\((\w+)\)")[0]
meses = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
mes_ref_num = mes_ref_texto.map(meses)

ano_divulgacao = datas_ipca["data_divulgacao"].dt.year
mes_divulgacao = datas_ipca["data_divulgacao"].dt.month

ano_ref = ano_divulgacao.where(mes_ref_num <= mes_divulgacao, ano_divulgacao - 1)

datas_ipca["DataReferencia"] = pd.to_datetime(
    ano_ref.astype(str) + "-" + mes_ref_num.astype(str) + "-01"
)

print(datas_ipca[["Release date", "DataReferencia", "data_divulgacao"]])

          Release date DataReferencia data_divulgacao
1   Jul 10, 2026 (Jun)     2026-06-01      2026-07-10
2   Jun 12, 2026 (May)     2026-05-01      2026-06-12
3   May 12, 2026 (Apr)     2026-04-01      2026-05-12
4   Apr 10, 2026 (Mar)     2026-03-01      2026-04-10
5   Mar 12, 2026 (Feb)     2026-02-01      2026-03-12
6   Feb 10, 2026 (Jan)     2026-01-01      2026-02-10
7   Jan 09, 2026 (Dec)     2025-12-01      2026-01-09
8   Dec 10, 2025 (Nov)     2025-11-01      2025-12-10
9   Nov 11, 2025 (Oct)     2025-10-01      2025-11-11
10  Oct 09, 2025 (Sep)     2025-09-01      2025-10-09
11  Sep 10, 2025 (Aug)     2025-08-01      2025-09-10
12  Aug 12, 2025 (Jul)     2025-07-01      2025-08-12
13  Jul 10, 2025 (Jun)     2025-06-01      2025-07-10
14  Jun 10, 2025 (May)     2025-05-01      2025-06-10
15  May 09, 2025 (Apr)     2025-04-01      2025-05-09
16  Apr 11, 2025 (Mar)     2025-03-01      2025-04-11
17  Mar 12, 2025 (Feb)     2025-02-01      2025-03-12
18  Feb 11, 2025 (Jan)     2

In [30]:
eventos_atualizado = pd.read_csv("../data/eventos.csv")

ipca_atualizado = eventos_atualizado[eventos_atualizado["indicador"] == "IPCA_BR"].copy()
ipca_atualizado["DataReferencia"] = pd.to_datetime(ipca_atualizado["data"])

ipca_atualizado = ipca_atualizado.merge(
    datas_ipca[["DataReferencia", "data_divulgacao"]],
    on="DataReferencia",
    how="left"
)

print(ipca_atualizado[["DataReferencia", "data_divulgacao", "actual", "forecast"]])
print(f"\nSem data de divulgação encontrada: {ipca_atualizado['data_divulgacao'].isna().sum()}")

   DataReferencia data_divulgacao  actual  forecast
0      2023-02-09             NaT    0.53    0.5550
1      2023-03-10             NaT    0.84    0.7800
2      2023-04-11             NaT    0.71    0.7700
3      2023-05-12             NaT    0.61    0.5500
4      2023-06-07             NaT    0.23    0.3700
5      2023-07-11             NaT   -0.08   -0.1000
6      2023-08-11             NaT    0.12    0.0600
7      2023-09-12             NaT    0.23    0.2600
8      2023-10-11             NaT    0.26    0.3500
9      2023-11-10             NaT    0.24    0.3500
10     2023-12-12             NaT    0.28    0.3256
11     2024-01-11             NaT    0.56    0.5200
12     2024-02-08             NaT    0.42    0.4200
13     2024-03-12             NaT    0.83    0.5000
14     2024-04-10             NaT    0.16    0.3350
15     2024-05-10             NaT    0.38    0.3500
16     2024-06-11             NaT    0.46    0.2600
17     2024-07-10             NaT    0.21    0.2100
18     2024-

In [31]:
ipca_atualizado = ipca_atualizado.dropna(subset=["data_divulgacao"])
ipca_atualizado["data"] = ipca_atualizado["data_divulgacao"].dt.strftime("%Y-%m-%d")
ipca_atualizado = ipca_atualizado[["indicador", "data", "actual", "forecast"]]

print(ipca_atualizado)

Empty DataFrame
Columns: [indicador, data, actual, forecast]
Index: []


In [32]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

ipca_completo = eventos_corrigido[eventos_corrigido["indicador"] == "IPCA_BR"].copy()
ipca_completo = calcular_surpresa(ipca_completo)
ipca_completo = calcular_ian(ipca_completo, termos_busca=["IPCA", "inflação"], geo="BR")
ipca_completo = calcular_ice(
    ipca_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

ipca_completo.to_csv("../data/eventos_ipca_completo.csv", index=False)
print(ipca_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-02-09        -0.118954  0.325843  0.456279
1  2023-03-10         0.285489  0.191011  0.409271
2  2023-04-11        -0.285489  0.561798  0.369906
3  2023-05-12         0.285489  0.303371  0.332035
4  2023-06-07        -0.666142  0.157303  0.293148
5  2023-07-11         0.095163  0.202247  0.287217
6  2023-08-11         0.285489  0.123596  0.202697
7  2023-09-12        -0.142745  0.314607  0.124736
8  2023-10-11        -0.428234  0.000000  0.127309
9  2023-11-10        -0.523397  0.134831  0.067917
10 2023-12-12        -0.216972  0.359551  0.046688
11 2024-01-11         0.190326  0.595506  0.736228
12 2024-02-08         0.000000  0.235955  0.288366
13 2024-03-12         1.570191  0.370787 -0.222174
14 2024-04-10        -0.832677  0.382022 -0.248402
15 2024-05-10         0.142745  0.247191  0.100423
16 2024-06-11         0.951631  0.449438 -0.479140
17 2024-07-10         0.000000  0.168539  0.072662
18 2024-08-09         0.856468 

In [33]:
import pandas as pd

cpi = pd.read_csv("../data/eventos_completo.csv")
ipca = pd.read_csv("../data/eventos_ipca_completo.csv")
selic = pd.read_csv("../data/eventos_selic_completo.csv")
payroll = pd.read_csv("../data/eventos_payroll_completo.csv")

eventos_todos = pd.concat([cpi, ipca, selic, payroll], ignore_index=True)
eventos_todos = eventos_todos.drop_duplicates(subset=["indicador", "data"])

eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)
print(eventos_todos["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
